In [2]:
# QUICK LOAD CELL: Copy & paste this into your new Stage 2 notebook
# Cell 1: Load all Stage 1 results

import pandas as pd
import numpy as np
import json
from datetime import datetime

# Load main datasets
df_raw = pd.read_csv('kcet_ml_project/data/df_optimized.csv')  # Using optimized as main
df_optimized = pd.read_csv('kcet_ml_project/data/df_optimized.csv')

# Load column definitions
with open('kcet_ml_project/configs/column_definitions.json', 'r') as f:
    cols = json.load(f)

NUMERIC_COLUMNS = cols['numeric_columns']
CATEGORICAL_COLUMNS = cols['categorical_columns']  
TARGET_COLUMN = cols['target_column']
GROUP_COLUMNS = cols['key_grouping_columns']

print(f"STAGE 2 DATA LOADED SUCCESSFULLY!")
print(f"   df_raw: {df_raw.shape}")
print(f"   df_optimized: {df_optimized.shape}")
print(f"   Ready for production preprocessing!")

# Quick verification
print(f"\nDATA VERIFICATION:")
print(f"   Numeric features: {len(NUMERIC_COLUMNS)}")
print(f"   Categorical features: {len(CATEGORICAL_COLUMNS)}")
print(f"   Target: {TARGET_COLUMN}")
print(f"   Year range: {df_optimized['Year'].min()}-{df_optimized['Year'].max()}")
print(f"   Crisis data available: {'Temporal_Regime' in df_optimized.columns}")


STAGE 2 DATA LOADED SUCCESSFULLY!
   df_raw: (270062, 23)
   df_optimized: (270062, 23)
   Ready for production preprocessing!

DATA VERIFICATION:
   Numeric features: 9
   Categorical features: 5
   Target: Cutoff_Rank
   Year range: 2020-2024
   Crisis data available: False


In [3]:
# STAGE 2 - STEP 2.1: BASIC CLEANING & CANONICALIZATION (FIXED)
# Cell 2: Text standardization, synonym mapping, and data canonicalization

import pandas as pd
import numpy as np
import re
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STAGE 2 - STEP 2.1: BASIC CLEANING & CANONICALIZATION")
print("=" * 80)

print(f"Starting with: {df_optimized.shape[0]:,} records, {df_optimized.shape[1]} features")

# ============================================================================
# 1. DATA TYPE FIXES & BASIC VALIDATION
# ============================================================================

print(f"\n🔧 STEP 2.1.0: DATA TYPE VALIDATION & FIXES")
print("="*60)

def fix_data_types(df):
    """Fix common data type issues that prevent processing"""
    df_fixed = df.copy()
    
    # Fix any list or object issues in categorical columns
    categorical_cols = ['College_Code', 'Branch', 'Category', 'Exam_Type', 'College_Tier']
    
    for col in categorical_cols:
        if col in df_fixed.columns:
            # Convert any non-string types to string
            df_fixed[col] = df_fixed[col].astype(str)
            
            # Handle 'nan' strings
            df_fixed[col] = df_fixed[col].replace('nan', np.nan)
    
    print(f"   ✅ Data types validated and fixed for {len([c for c in categorical_cols if c in df_fixed.columns])} columns")
    return df_fixed

# Apply data type fixes
df_stage2_clean = fix_data_types(df_optimized)

# ============================================================================
# 2. TEXT STANDARDIZATION TRANSFORMER
# ============================================================================

print(f"\n🔧 STEP 2.1.1: TEXT STANDARDIZATION")
print("="*60)

class TextStandardizer(BaseEstimator, TransformerMixin):
    """Standardize text columns with consistent formatting"""
    
    def __init__(self, text_columns=None, title_case=True):
        self.text_columns = text_columns or []
        self.title_case = title_case
        self.standardization_map = {}
    
    def fit(self, X, y=None):
        """Learn standardization patterns from training data"""
        # Build standardization maps during fit
        for col in self.text_columns:
            if col in X.columns:
                try:
                    # Get unique values and create standardized versions
                    unique_values = X[col].dropna().unique()
                    standardized = {}
                    
                    for value in unique_values:
                        # Basic cleaning: strip whitespace, normalize case
                        cleaned = str(value).strip()
                        
                        if self.title_case:
                            cleaned = cleaned.title()
                        else:
                            cleaned = cleaned.upper()
                        
                        # Handle multiple spaces
                        cleaned = re.sub(r'\s+', ' ', cleaned)
                        
                        standardized[str(value)] = cleaned
                    
                    self.standardization_map[col] = standardized
                except Exception as e:
                    print(f"   ⚠️ Warning: Could not standardize {col}: {str(e)}")
                    self.standardization_map[col] = {}
        
        print(f"   ✅ Text standardizer fitted for {len([c for c in self.text_columns if c in self.standardization_map])} columns")
        return self
    
    def transform(self, X):
        """Apply text standardization"""
        X_clean = X.copy()
        
        for col in self.text_columns:
            if col in X_clean.columns and col in self.standardization_map:
                try:
                    # Convert to string first, then map
                    X_clean[col] = X_clean[col].astype(str)
                    X_clean[col] = X_clean[col].map(self.standardization_map[col]).fillna(X_clean[col])
                except Exception as e:
                    print(f"   ⚠️ Warning: Could not transform {col}: {str(e)}")
        
        return X_clean

# Apply text standardization
text_columns = ['College_Name', 'Branch', 'Category']
available_text_cols = [col for col in text_columns if col in df_stage2_clean.columns]

if available_text_cols:
    text_standardizer = TextStandardizer(text_columns=available_text_cols, title_case=True)
    df_stage2_clean = text_standardizer.fit_transform(df_stage2_clean)
    print(f"Text standardization applied to: {available_text_cols}")

# ============================================================================
# 3. BRANCH SYNONYM MAPPING
# ============================================================================

print(f"\n🔧 STEP 2.1.2: BRANCH SYNONYM MAPPING")
print("="*60)

class BranchSynonymMapper(BaseEstimator, TransformerMixin):
    """Map branch synonyms and variations to canonical names"""
    
    def __init__(self):
        # Comprehensive branch synonym dictionary for engineering
        self.branch_synonyms = {
            # Computer Science variations
            'Computer Science': ['CS', 'CSE', 'COMPUTER SCIENCE', 'Computer Science Engineering', 
                               'Computer Science & Engineering', 'Comp Sci', 'Computers', 'CS Computers'],
            
            # Electronics variations  
            'Electronics': ['EC', 'ECE', 'ELECTRONICS', 'Electronics Engineering',
                          'Electronics & Communication', 'Electronics & Comm', 'E&C', 'EC Electronics'],
            
            # Mechanical variations
            'Mechanical': ['ME', 'MECHANICAL', 'Mechanical Engineering', 'Mech', 'Mechanical Engg', 'ME Mechanical'],
            
            # Civil variations
            'Civil': ['CV', 'CE', 'CIVIL', 'Civil Engineering', 'Civil Engg', 'CV Civil'],
            
            # Electrical variations
            'Electrical': ['EE', 'ELECTRICAL', 'Electrical Engineering', 'Electrical Engg', 'E&E', 'EE Electrical'],
            
            # Information Science variations
            'Information Science': ['IS', 'ISE', 'INFORMATION SCIENCE', 'Info Science',
                                  'Information Science & Engineering', 'IS&E', 'IE Information Science'],
            
            # Information Technology variations
            'Information Technology': ['IT', 'INFORMATION TECHNOLOGY', 'Info Tech', 'IT Engg', 'IT Information Technology'],
            
            # Chemical variations
            'Chemical': ['CH', 'CHEMICAL', 'Chemical Engineering', 'Chem', 'Chemical Engg', 'CH Chemical'],
            
            # Aeronautical variations
            'Aeronautical': ['AE', 'AERONAUTICAL', 'Aeronautical Engineering', 'Aero', 'AE Aeronautical'],
            
            # Biotechnology variations
            'Biotechnology': ['BT', 'BIOTECHNOLOGY', 'Biotech', 'Bio Technology', 'BT Biotechnology']
        }
        
        # Create reverse mapping for lookup
        self.synonym_to_canonical = {}
        for canonical, synonyms in self.branch_synonyms.items():
            # Add canonical name to itself
            self.synonym_to_canonical[canonical.upper()] = canonical
            # Add all synonyms mapping to canonical
            for synonym in synonyms:
                self.synonym_to_canonical[synonym.upper()] = canonical
    
    def fit(self, X, y=None):
        """Fit is not needed for rule-based mapping"""
        print(f"   ✅ Branch synonym mapper initialized with {len(self.branch_synonyms)} canonical branches")
        return self
    
    def transform(self, X):
        """Apply branch synonym mapping"""
        X_mapped = X.copy()
        
        if 'Branch' in X_mapped.columns:
            def map_branch(branch_name):
                if pd.isna(branch_name):
                    return branch_name
                
                # Convert to string and clean
                branch_clean = str(branch_name).upper().strip()
                
                # Try exact match first
                if branch_clean in self.synonym_to_canonical:
                    return self.synonym_to_canonical[branch_clean]
                
                # Try partial matching for complex cases
                for synonym, canonical in self.synonym_to_canonical.items():
                    if synonym in branch_clean or branch_clean in synonym:
                        return canonical
                
                # Return cleaned original if no match found
                return str(branch_name).title()
            
            X_mapped['Branch'] = X_mapped['Branch'].apply(map_branch)
        
        return X_mapped

# Apply branch synonym mapping
branch_mapper = BranchSynonymMapper()
df_stage2_clean = branch_mapper.fit_transform(df_stage2_clean)

# Show mapping results safely
if 'Branch' in df_stage2_clean.columns:
    try:
        branch_counts_before = df_optimized['Branch'].value_counts()
        branch_counts_after = df_stage2_clean['Branch'].value_counts()
        
        print(f"Branch consolidation results:")
        print(f"   Before: {len(branch_counts_before)} unique branches")
        print(f"   After: {len(branch_counts_after)} unique branches")
        print(f"   Consolidated: {len(branch_counts_before) - len(branch_counts_after)} branches")

        # Show top branches after mapping
        print(f"\nTop 5 branches after standardization:")
        for i, (branch, count) in enumerate(branch_counts_after.head().items(), 1):
            print(f"   {i}. {branch}: {count:,} records")
    except Exception as e:
        print(f"   ⚠️ Could not show branch mapping results: {str(e)}")

# ============================================================================
# 4. CATEGORY NORMALIZATION
# ============================================================================

print(f"\n🔧 STEP 2.1.3: CATEGORY NORMALIZATION")
print("="*60)

class CategoryNormalizer(BaseEstimator, TransformerMixin):
    """Normalize category codes and handle variations"""
    
    def __init__(self):
        # Karnataka reservation category patterns
        self.category_patterns = {
            # General categories
            'General': ['1G', '1K', '1R', 'GM', 'GENERAL'],
            
            # OBC categories
            'OBC_2A': ['2A', '2AG', '2AK', '2AR', 'OBC-2A'],
            'OBC_2B': ['2B', '2BG', '2BK', '2BR', 'OBC-2B'],
            'OBC_3A': ['3A', '3AG', '3AK', '3AR', 'OBC-3A'],
            'OBC_3B': ['3B', '3BG', '3BK', '3BR', 'OBC-3B'],
            
            # SC categories
            'SC': ['SC', 'SCG', 'SCK', 'SCR', 'SCHEDULED_CASTE'],
            
            # ST categories  
            'ST': ['ST', 'STG', 'STK', 'STR', 'SCHEDULED_TRIBE'],
            
            # Special categories
            'Category_I': ['CA', 'CAG', 'CAK', 'CAR', 'CAT_I'],
            'Category_II': ['CB', 'CBG', 'CBK', 'CBR', 'CAT_II']
        }
        
        # Create lookup dictionary
        self.category_lookup = {}
        for normalized, variations in self.category_patterns.items():
            for variation in variations:
                self.category_lookup[variation.upper()] = normalized
    
    def fit(self, X, y=None):
        """Fit normalizer"""
        print(f"   ✅ Category normalizer initialized with {len(self.category_patterns)} normalized categories")
        return self
    
    def transform(self, X):
        """Apply category normalization"""
        X_normalized = X.copy()
        
        if 'Category' in X_normalized.columns:
            def normalize_category(category):
                if pd.isna(category):
                    return 'Unknown'
                
                category_clean = str(category).upper().strip()
                
                # Direct lookup
                if category_clean in self.category_lookup:
                    return self.category_lookup[category_clean]
                
                # Pattern matching for complex categories
                for normalized, variations in self.category_patterns.items():
                    if any(v in category_clean for v in variations):
                        return normalized
                
                return 'Other'  # For unrecognized categories
            
            X_normalized['Category'] = X_normalized['Category'].apply(normalize_category)
            
            # Also create simplified category
            def simplify_category(category):
                if pd.isna(category) or category == 'Unknown':
                    return 'Unknown'
                elif 'OBC' in str(category):
                    return 'OBC'
                elif str(category) in ['SC', 'ST']:
                    return str(category)
                elif 'General' in str(category):
                    return 'General'
                else:
                    return 'Other'
            
            X_normalized['Category_Simplified'] = X_normalized['Category'].apply(simplify_category)
        
        return X_normalized

# Apply category normalization
category_normalizer = CategoryNormalizer()
df_stage2_clean = category_normalizer.fit_transform(df_stage2_clean)

# Show normalization results safely
if 'Category' in df_stage2_clean.columns:
    try:
        cat_before = df_optimized['Category'].astype(str).value_counts()
        cat_after = df_stage2_clean['Category'].value_counts()
        
        print(f"Category normalization results:")
        print(f"   Before: {len(cat_before)} unique categories")
        print(f"   After: {len(cat_after)} unique categories")
        
        print(f"\nNormalized categories:")
        for cat, count in cat_after.items():
            pct = count / len(df_stage2_clean) * 100
            print(f"   {cat}: {count:,} ({pct:.1f}%)")
    except Exception as e:
        print(f"   ⚠️ Could not show category results: {str(e)}")

# ============================================================================
# 5. ROUND NORMALIZATION
# ============================================================================

print(f"\n🔧 STEP 2.1.4: ROUND NORMALIZATION")
print("="*60)

class RoundNormalizer(BaseEstimator, TransformerMixin):
    """Normalize round values to integer sequence"""
    
    def fit(self, X, y=None):
        """Fit normalizer"""
        print(f"   ✅ Round normalizer fitted")
        return self
    
    def transform(self, X):
        """Apply round normalization"""
        X_rounded = X.copy()
        
        if 'Round' in X_rounded.columns:
            try:
                # Ensure Round is integer
                X_rounded['Round'] = pd.to_numeric(X_rounded['Round'], errors='coerce').fillna(1).astype(int)
                
                # Create round sequence features
                X_rounded['Round_Sequence'] = X_rounded['Round']
                X_rounded['Is_Final_Round'] = (X_rounded['Round'] == X_rounded['Round'].max()).astype(int)
                X_rounded['Is_First_Round'] = (X_rounded['Round'] == X_rounded['Round'].min()).astype(int)
                
                print(f"   ✅ Round normalization applied")
            except Exception as e:
                print(f"   ⚠️ Could not normalize rounds: {str(e)}")
        
        return X_rounded

# Apply round normalization
round_normalizer = RoundNormalizer()
df_stage2_clean = round_normalizer.fit_transform(df_stage2_clean)

if 'Round' in df_stage2_clean.columns:
    try:
        round_dist = df_stage2_clean['Round'].value_counts().sort_index()
        print(f"   Round distribution: {dict(round_dist)}")
    except:
        print(f"   ✅ Round processing complete")

# ============================================================================
# 6. DATA QUALITY ASSESSMENT
# ============================================================================

print(f"\n📊 STEP 2.1.5: DATA QUALITY ASSESSMENT")
print("="*60)

# Compare data quality before and after cleaning
print(f"CLEANING RESULTS SUMMARY:")
print(f"   Records: {len(df_stage2_clean):,} (no loss)")
print(f"   Features: {df_optimized.shape[1]} → {df_stage2_clean.shape[1]} (+{df_stage2_clean.shape[1] - df_optimized.shape[1]} new)")

# Missing values comparison
try:
    missing_before = df_optimized.isnull().sum().sum()
    missing_after = df_stage2_clean.isnull().sum().sum()
    print(f"   Missing values: {missing_before} → {missing_after}")
except:
    print(f"   Missing values: assessed")

# New features created
new_features = set(df_stage2_clean.columns) - set(df_optimized.columns)
if new_features:
    print(f"   New features created: {list(new_features)}")

# ============================================================================
# 7. VALIDATION & NEXT STEPS
# ============================================================================

print(f"\n✅ STEP 2.1: BASIC CLEANING & CANONICALIZATION COMPLETE!")
print("="*80)

print(f"ACCOMPLISHED:")
print(f"   ✅ Data types validated and fixed")
print(f"   ✅ Text standardization applied")
print(f"   ✅ Branch synonym mapping completed") 
print(f"   ✅ Category normalization finished")
print(f"   ✅ Round values normalized")
print(f"   ✅ Data quality maintained")

print(f"\nCLEANED DATASET READY:")
print(f"   📊 Shape: {df_stage2_clean.shape}")
print(f"   🔧 Standardized text fields")
print(f"   📝 Normalized categorical values")
print(f"   ⚙️ Ready for Step 2.2: Missing Value Handling")

# Quick sample of cleaned data
print(f"\n👀 CLEANED DATA SAMPLE:")
sample_cols = ['Year', 'College_Code', 'Branch', 'Category', 'Round', 'Cutoff_Rank']
if 'Category_Simplified' in df_stage2_clean.columns:
    sample_cols.insert(-1, 'Category_Simplified')

available_sample_cols = [col for col in sample_cols if col in df_stage2_clean.columns]
try:
    print(df_stage2_clean[available_sample_cols].head(3).to_string(index=False))
except Exception as e:
    print("   Sample data display unavailable due to data complexity")

print(f"\n🚀 READY FOR STEP 2.2: MISSING VALUE HANDLING!")

# Store the cleaned dataset for next step
df_cleaned = df_stage2_clean.copy()


STAGE 2 - STEP 2.1: BASIC CLEANING & CANONICALIZATION
Starting with: 270,062 records, 23 features

🔧 STEP 2.1.0: DATA TYPE VALIDATION & FIXES
   ✅ Data types validated and fixed for 5 columns

🔧 STEP 2.1.1: TEXT STANDARDIZATION
   ✅ Text standardizer fitted for 3 columns
Text standardization applied to: ['College_Name', 'Branch', 'Category']

🔧 STEP 2.1.2: BRANCH SYNONYM MAPPING
   ✅ Branch synonym mapper initialized with 10 canonical branches
Branch consolidation results:
   Before: 418 unique branches
   After: 33 unique branches
   Consolidated: 385 branches

Top 5 branches after standardization:
   1. Computer Science: 86,913 records
   2. Electronics: 73,364 records
   3. Civil: 49,282 records
   4. Electrical: 21,542 records
   5. Mechanical: 14,808 records

🔧 STEP 2.1.3: CATEGORY NORMALIZATION
   ✅ Category normalizer initialized with 9 normalized categories
Category normalization results:
   Before: 53 unique categories
   After: 8 unique categories

Normalized categories:
   G

In [4]:
# STAGE 2 - STEP 2.2: MISSING VALUE HANDLING (FIXED)
# Cell 3: Production-grade imputation strategies with robust alternatives

import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Enable experimental IterativeImputer (optional - with fallback)
try:
    from sklearn.experimental import enable_iterative_imputer
    from sklearn.impute import IterativeImputer
    ITERATIVE_AVAILABLE = True
except ImportError:
    ITERATIVE_AVAILABLE = False
    print("IterativeImputer not available - using robust alternatives")

import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STAGE 2 - STEP 2.2: MISSING VALUE HANDLING")
print("=" * 80)

print(f"Starting with cleaned data: {df_cleaned.shape[0]:,} records, {df_cleaned.shape[1]} features")

# ============================================================================
# 1. MISSING VALUE ANALYSIS
# ============================================================================

print(f"\n🔍 STEP 2.2.1: COMPREHENSIVE MISSING VALUE ANALYSIS")
print("="*60)

def analyze_missing_values(df):
    """Comprehensive missing value analysis"""
    
    missing_info = []
    total_records = len(df)
    
    for column in df.columns:
        missing_count = df[column].isnull().sum()
        missing_pct = (missing_count / total_records) * 100
        
        if missing_count > 0:
            # Analyze missing patterns
            dtype = str(df[column].dtype)
            unique_count = df[column].nunique()
            
            missing_info.append({
                'column': column,
                'missing_count': missing_count,
                'missing_pct': missing_pct,
                'dtype': dtype,
                'unique_values': unique_count
            })
    
    return pd.DataFrame(missing_info)

missing_analysis = analyze_missing_values(df_cleaned)

print("MISSING VALUE ANALYSIS:")
if len(missing_analysis) > 0:
    print(f"Columns with missing values: {len(missing_analysis)}")
    print("\nDetailed breakdown:")
    for _, row in missing_analysis.iterrows():
        print(f"  {row['column']}: {row['missing_count']:,} ({row['missing_pct']:.2f}%) - {row['dtype']}")
else:
    print("✅ No missing values detected!")

# Check missing value patterns
print(f"\nMISSING PATTERNS ANALYSIS:")
missing_combinations = df_cleaned.isnull().sum(axis=1).value_counts().sort_index()
print("Records by number of missing features:")
total_complete = missing_combinations.get(0, 0)
print(f"  Complete records: {total_complete:,} ({total_complete/len(df_cleaned)*100:.1f}%)")

for missing_count, record_count in missing_combinations.items():
    if missing_count > 0:
        pct = record_count / len(df_cleaned) * 100
        print(f"  {missing_count} missing: {record_count:,} records ({pct:.2f}%)")

# ============================================================================
# 2. ROBUST MISSING VALUE IMPUTATION STRATEGIES
# ============================================================================

print(f"\n🔧 STEP 2.2.2: ROBUST IMPUTATION STRATEGIES")
print("="*60)

class RobustMissingValueHandler(BaseEstimator, TransformerMixin):
    """Robust missing value handler with multiple strategies"""
    
    def __init__(self):
        self.numeric_imputers = {}
        self.categorical_imputers = {}
        self.missing_indicators = {}
        self.imputation_stats = {}
        self.group_imputers = {}  # For group-based imputation
        
    def fit(self, X, y=None):
        """Fit imputation strategies based on data characteristics"""
        
        self.feature_types = self._identify_feature_types(X)
        
        print(f"   Feature type identification:")
        for ftype, cols in self.feature_types.items():
            if cols:
                print(f"     {ftype}: {len(cols)} features")
        
        # Fit numeric imputers
        for col in self.feature_types['numeric']:
            if col in X.columns and X[col].isnull().sum() > 0:
                missing_pct = X[col].isnull().sum() / len(X) * 100
                
                if missing_pct < 5:
                    # Very low missingness: use median
                    imputer = SimpleImputer(strategy='median')
                    strategy = 'median'
                elif missing_pct < 15:
                    # Low missingness: use mean for normal distributions, median otherwise
                    if self._is_roughly_normal(X[col].dropna()):
                        imputer = SimpleImputer(strategy='mean')
                        strategy = 'mean'
                    else:
                        imputer = SimpleImputer(strategy='median')
                        strategy = 'median'
                elif missing_pct < 40:
                    # Medium missingness: try group-based imputation
                    if self._setup_group_imputation(X, col):
                        strategy = 'group_median'
                    else:
                        imputer = SimpleImputer(strategy='median')
                        strategy = 'median_fallback'
                else:
                    # High missingness: use robust median
                    imputer = SimpleImputer(strategy='median')
                    strategy = 'median_robust'
                
                # Fit imputer if not using group-based
                if strategy != 'group_median':
                    available_data = X[[col]].dropna()
                    if len(available_data) > 0:
                        imputer.fit(available_data)
                        self.numeric_imputers[col] = imputer
                
                # Store statistics
                if len(X[col].dropna()) > 0:
                    self.imputation_stats[col] = {
                        'strategy': strategy,
                        'missing_pct': missing_pct,
                        'fill_value': X[col].median(),
                        'mean_value': X[col].mean(),
                        'std_value': X[col].std()
                    }
        
        # Fit categorical imputers
        for col in self.feature_types['categorical']:
            if col in X.columns and X[col].isnull().sum() > 0:
                missing_pct = X[col].isnull().sum() / len(X) * 100
                
                if missing_pct < 10:
                    # Low missingness: use most frequent
                    imputer = SimpleImputer(strategy='most_frequent')
                    strategy = 'mode'
                elif missing_pct < 30:
                    # Medium missingness: use mode but create missing category
                    imputer = SimpleImputer(strategy='most_frequent')
                    strategy = 'mode_with_indicator'
                else:
                    # High missingness: explicit missing category
                    imputer = SimpleImputer(strategy='constant', fill_value='MISSING')
                    strategy = 'missing_category'
                
                # Fit imputer
                non_null_data = X[[col]].dropna()
                if len(non_null_data) > 0:
                    imputer.fit(non_null_data)
                    self.categorical_imputers[col] = imputer
                    
                    mode_value = X[col].mode().iloc[0] if len(X[col].mode()) > 0 else 'UNKNOWN'
                    self.imputation_stats[col] = {
                        'strategy': strategy,
                        'missing_pct': missing_pct,
                        'mode_value': mode_value,
                        'unique_count': X[col].nunique()
                    }
        
        # Create missing indicators for important features
        important_features = ['Cutoff_Rank', 'Historical_Mean_Primary', 'Category_Score', 'Branch_Popularity']
        for col in X.columns:
            missing_pct = X[col].isnull().sum() / len(X) * 100
            if (missing_pct > 5 and missing_pct < 95) or col in important_features:
                if X[col].isnull().sum() > 0:
                    self.missing_indicators[f'{col}_was_missing'] = col
        
        print(f"   ✅ Fitted imputers: {len(self.numeric_imputers)} numeric, {len(self.categorical_imputers)} categorical")
        print(f"   ✅ Group imputers: {len(self.group_imputers)} features")
        print(f"   ✅ Missing indicators: {len(self.missing_indicators)} created")
        
        return self
    
    def transform(self, X):
        """Apply robust imputation strategies"""
        X_imputed = X.copy()
        
        # Create missing indicators first (before imputation)
        for indicator_name, original_col in self.missing_indicators.items():
            if original_col in X_imputed.columns:
                X_imputed[indicator_name] = X_imputed[original_col].isnull().astype(int)
        
        # Apply numeric imputation
        for col, imputer in self.numeric_imputers.items():
            if col in X_imputed.columns:
                try:
                    # Use fitted imputer
                    X_imputed[col] = imputer.transform(X_imputed[[col]]).ravel()
                except Exception as e:
                    # Fallback to statistical imputation
                    if col in self.imputation_stats:
                        fill_value = self.imputation_stats[col]['fill_value']
                        X_imputed[col] = X_imputed[col].fillna(fill_value)
        
        # Apply group-based imputation
        for col, group_info in self.group_imputers.items():
            if col in X_imputed.columns:
                X_imputed = self._apply_group_imputation(X_imputed, col, group_info)
        
        # Apply categorical imputation
        for col, imputer in self.categorical_imputers.items():
            if col in X_imputed.columns:
                try:
                    X_imputed[col] = imputer.transform(X_imputed[[col]]).ravel()
                except Exception as e:
                    # Fallback to mode
                    if col in self.imputation_stats:
                        mode_value = self.imputation_stats[col]['mode_value']
                        X_imputed[col] = X_imputed[col].fillna(mode_value)
                    else:
                        X_imputed[col] = X_imputed[col].fillna('UNKNOWN')
        
        return X_imputed
    
    def _identify_feature_types(self, X):
        """Identify feature types for appropriate imputation"""
        feature_types = {
            'numeric': [],
            'categorical': [],
            'boolean': []
        }
        
        for col in X.columns:
            if X[col].dtype in ['int64', 'float64']:
                # Check if it's actually categorical (few unique values relative to size)
                unique_count = X[col].nunique()
                total_count = len(X[col].dropna())
                
                if unique_count <= 10 and col not in ['Year', 'Cutoff_Rank']:
                    feature_types['boolean'].append(col)
                elif unique_count / total_count < 0.05 and unique_count <= 50:
                    feature_types['categorical'].append(col)
                else:
                    feature_types['numeric'].append(col)
            elif X[col].dtype == 'bool':
                feature_types['boolean'].append(col)
            else:
                feature_types['categorical'].append(col)
        
        return feature_types
    
    def _is_roughly_normal(self, series):
        """Simple normality check using skewness"""
        try:
            skewness = series.skew()
            return abs(skewness) < 1.0  # Rough threshold for normality
        except:
            return False
    
    def _setup_group_imputation(self, X, col):
        """Setup group-based imputation for missing values"""
        # Try to find good grouping columns
        potential_groups = ['College_Code', 'Branch', 'Category', 'Year']
        available_groups = [g for g in potential_groups if g in X.columns]
        
        if len(available_groups) >= 2:
            # Use first two available grouping columns
            group_cols = available_groups[:2]
            
            # Check if group-based imputation makes sense
            group_stats = X.groupby(group_cols)[col].agg(['count', 'median', 'std']).reset_index()
            valid_groups = group_stats[group_stats['count'] >= 5]  # At least 5 observations per group
            
            if len(valid_groups) > 10:  # At least 10 valid groups
                self.group_imputers[col] = {
                    'group_cols': group_cols,
                    'group_stats': dict(zip(
                        [tuple(row[group_cols]) for _, row in valid_groups.iterrows()],
                        valid_groups['median']
                    )),
                    'global_median': X[col].median()
                }
                return True
        
        return False
    
    def _apply_group_imputation(self, X, col, group_info):
        """Apply group-based imputation"""
        group_cols = group_info['group_cols']
        group_stats = group_info['group_stats']
        global_median = group_info['global_median']
        
        # Create a copy to avoid modifying original
        X_copy = X.copy()
        
        # Find missing values
        missing_mask = X_copy[col].isnull()
        
        if missing_mask.sum() > 0:
            # For each missing value, try to impute based on group
            for idx in X_copy[missing_mask].index:
                group_key = tuple(X_copy.loc[idx, group_cols])
                
                if group_key in group_stats:
                    X_copy.loc[idx, col] = group_stats[group_key]
                else:
                    X_copy.loc[idx, col] = global_median
        
        return X_copy
    
    def get_imputation_summary(self):
        """Get summary of imputation strategies used"""
        return pd.DataFrame.from_dict(self.imputation_stats, orient='index')

# ============================================================================
# 3. APPLY ROBUST MISSING VALUE HANDLING
# ============================================================================

print(f"\n🚀 STEP 2.2.3: APPLYING ROBUST IMPUTATION")
print("="*60)

# Initialize and fit the missing value handler
mv_handler = RobustMissingValueHandler()
df_imputed = mv_handler.fit_transform(df_cleaned)

print(f"\nImputation completed!")

# Show imputation summary
if len(mv_handler.imputation_stats) > 0:
    imputation_summary = mv_handler.get_imputation_summary()
    print(f"\nIMPUTATION SUMMARY:")
    for col, stats in mv_handler.imputation_stats.items():
        strategy = stats.get('strategy', 'unknown')
        missing_pct = stats.get('missing_pct', 0)
        print(f"   {col}: {strategy} strategy ({missing_pct:.1f}% missing)")

# ============================================================================
# 4. VALIDATION & QUALITY CHECK
# ============================================================================

print(f"\n📊 STEP 2.2.4: IMPUTATION QUALITY VALIDATION")
print("="*60)

# Before/after comparison
missing_before = df_cleaned.isnull().sum().sum()
missing_after = df_imputed.isnull().sum().sum()

print(f"IMPUTATION RESULTS:")
print(f"   Missing values before: {missing_before:,}")
print(f"   Missing values after: {missing_after:,}")

if missing_before > 0:
    reduction_pct = (missing_before - missing_after) / missing_before * 100
    print(f"   Reduction: {missing_before - missing_after:,} ({reduction_pct:.1f}%)")

# Check for any remaining missing values
remaining_missing = df_imputed.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0]

if len(remaining_missing) > 0:
    print(f"\nREMAINING MISSING VALUES:")
    for col, count in remaining_missing.items():
        pct = count / len(df_imputed) * 100
        print(f"   {col}: {count:,} ({pct:.2f}%)")
else:
    print(f"\n✅ NO REMAINING MISSING VALUES!")

# Data quality checks
print(f"\nDATA QUALITY AFTER IMPUTATION:")
print(f"   Records: {len(df_imputed):,} (no loss)")
print(f"   Features: {df_cleaned.shape[1]} → {df_imputed.shape[1]} (+{df_imputed.shape[1] - df_cleaned.shape[1]} indicators)")

# ============================================================================
# 5. ADVANCED MISSING VALUE FEATURES
# ============================================================================

print(f"\n🔧 STEP 2.2.5: ADVANCED MISSING VALUE FEATURES")
print("="*60)

# Create missingness pattern features
def create_missing_patterns(df_before, df_after):
    """Create features based on missing patterns"""
    
    # Total missing count per record
    df_after['missing_count_total'] = df_before.isnull().sum(axis=1)
    
    # Missing rate per record
    df_after['missing_rate'] = df_after['missing_count_total'] / len(df_before.columns)
    
    # Critical feature missing indicator
    critical_features = ['Cutoff_Rank', 'College_Code', 'Branch', 'Category']
    available_critical = [col for col in critical_features if col in df_before.columns]
    
    if available_critical:
        df_after['critical_missing'] = df_before[available_critical].isnull().any(axis=1).astype(int)
    else:
        df_after['critical_missing'] = 0
    
    return df_after

# Apply missing pattern features
df_imputed = create_missing_patterns(df_cleaned, df_imputed)

print(f"Advanced missing features created:")
pattern_features = ['missing_count_total', 'missing_rate', 'critical_missing']
for feature in pattern_features:
    if feature in df_imputed.columns:
        stats = df_imputed[feature].describe()
        print(f"   {feature}: mean={stats['mean']:.3f}, max={stats['max']:.3f}")

# ============================================================================
# 6. FINAL VALIDATION & SUMMARY
# ============================================================================

print(f"\n✅ STEP 2.2: MISSING VALUE HANDLING COMPLETE!")
print("="*80)

print(f"ACCOMPLISHED:")
print(f"   ✅ Comprehensive missing value analysis")
print(f"   ✅ Robust imputation strategies (no experimental dependencies)")
print(f"   ✅ Group-based imputation for complex patterns")
print(f"   ✅ Missing value indicators created")
print(f"   ✅ Missing pattern features engineered")
print(f"   ✅ Data quality validation passed")

print(f"\nFINAL DATASET STATISTICS:")
print(f"   📊 Shape: {df_imputed.shape}")
print(f"   🔍 Missing values: {df_imputed.isnull().sum().sum()}")

# Calculate data completeness
total_cells = len(df_imputed) * len(df_imputed.columns)
missing_cells = df_imputed.isnull().sum().sum()
completeness = ((total_cells - missing_cells) / total_cells * 100)
print(f"   🎯 Data completeness: {completeness:.2f}%")

# Sample of final data
print(f"\n👀 FINAL IMPUTED DATA SAMPLE:")
sample_cols = ['Year', 'College_Code', 'Branch', 'Category', 'Cutoff_Rank', 'missing_count_total']
available_sample_cols = [col for col in sample_cols if col in df_imputed.columns]
try:
    print(df_imputed[available_sample_cols].head(3).to_string(index=False))
except:
    print("   Sample display shows data is properly formatted")

print(f"\n🚀 READY FOR STEP 2.3: TARGET TRANSFORMATIONS!")

# Store final dataset for next step
df_imputed_final = df_imputed.copy()


STAGE 2 - STEP 2.2: MISSING VALUE HANDLING
Starting with cleaned data: 270,062 records, 26 features

🔍 STEP 2.2.1: COMPREHENSIVE MISSING VALUE ANALYSIS
MISSING VALUE ANALYSIS:
Columns with missing values: 2

Detailed breakdown:
  Historical_Std_Raw: 1,800 (0.67%) - float64
  Volatility_Category: 1,800 (0.67%) - object

MISSING PATTERNS ANALYSIS:
Records by number of missing features:
  Complete records: 268,262 (99.3%)
  2 missing: 1,800 records (0.67%)

🔧 STEP 2.2.2: ROBUST IMPUTATION STRATEGIES

🚀 STEP 2.2.3: APPLYING ROBUST IMPUTATION
   Feature type identification:
     numeric: 5 features
     categorical: 14 features
     boolean: 7 features
   ✅ Fitted imputers: 1 numeric, 1 categorical
   ✅ Group imputers: 0 features
   ✅ Missing indicators: 0 created

Imputation completed!

IMPUTATION SUMMARY:
   Historical_Std_Raw: median strategy (0.7% missing)
   Volatility_Category: mode strategy (0.7% missing)

📊 STEP 2.2.4: IMPUTATION QUALITY VALIDATION
IMPUTATION RESULTS:
   Missing val

In [5]:
# STAGE 2 - STEP 2.3: TARGET TRANSFORMATIONS & ALTERNATIVE TASKS
# Cell 4: Advanced target engineering for optimal model performance

import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import PowerTransformer, QuantileTransformer
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STAGE 2 - STEP 2.3: TARGET TRANSFORMATIONS & ALTERNATIVE TASKS")
print("=" * 80)

print(f"Starting with imputed data: {df_imputed_final.shape[0]:,} records, {df_imputed_final.shape[1]} features")

# ============================================================================
# 1. TARGET VARIABLE ANALYSIS
# ============================================================================

print(f"\n🔍 STEP 2.3.1: TARGET VARIABLE COMPREHENSIVE ANALYSIS")
print("="*60)

def analyze_target_distribution(df, target_col='Cutoff_Rank'):
    """Comprehensive target variable analysis"""
    
    if target_col not in df.columns:
        print(f"❌ Target column {target_col} not found")
        return None
    
    target = df[target_col].copy()
    
    # Basic statistics
    stats_dict = {
        'count': len(target),
        'mean': target.mean(),
        'median': target.median(),
        'std': target.std(),
        'min': target.min(),
        'max': target.max(),
        'skewness': target.skew(),
        'kurtosis': target.kurtosis(),
        'range': target.max() - target.min(),
        'iqr': target.quantile(0.75) - target.quantile(0.25)
    }
    
    print(f"TARGET VARIABLE ANALYSIS ({target_col}):")
    print(f"   Count: {stats_dict['count']:,}")
    print(f"   Mean: {stats_dict['mean']:,.0f}")
    print(f"   Median: {stats_dict['median']:,.0f}")
    print(f"   Std Dev: {stats_dict['std']:,.0f}")
    print(f"   Range: {stats_dict['min']:,.0f} - {stats_dict['max']:,.0f}")
    print(f"   IQR: {stats_dict['iqr']:,.0f}")
    print(f"   Skewness: {stats_dict['skewness']:.3f}")
    print(f"   Kurtosis: {stats_dict['kurtosis']:.3f}")
    
    # Distribution assessment
    print(f"\nDISTRIBUTION CHARACTERISTICS:")
    if abs(stats_dict['skewness']) < 0.5:
        skew_assessment = "Normal"
    elif abs(stats_dict['skewness']) < 1:
        skew_assessment = "Moderate"
    else:
        skew_assessment = "High"
    
    print(f"   Skewness: {skew_assessment} ({'Right' if stats_dict['skewness'] > 0 else 'Left'} skewed)")
    
    if stats_dict['kurtosis'] < 3:
        kurt_assessment = "Platykurtic (flatter)"
    elif stats_dict['kurtosis'] > 3:
        kurt_assessment = "Leptokurtic (peaked)"
    else:
        kurt_assessment = "Mesokurtic (normal)"
    
    print(f"   Kurtosis: {kurt_assessment}")
    
    # Outlier detection
    q1 = target.quantile(0.25)
    q3 = target.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = target[(target < lower_bound) | (target > upper_bound)]
    outlier_pct = len(outliers) / len(target) * 100
    
    print(f"   Outliers: {len(outliers):,} ({outlier_pct:.2f}%)")
    print(f"   Normal range: {lower_bound:,.0f} - {upper_bound:,.0f}")
    
    return stats_dict

# Analyze current target
target_stats = analyze_target_distribution(df_imputed_final, 'Cutoff_Rank')

# ============================================================================
# 2. TARGET TRANSFORMATION STRATEGIES
# ============================================================================

print(f"\n🔧 STEP 2.3.2: TARGET TRANSFORMATION STRATEGIES")
print("="*60)

class AdvancedTargetTransformer(BaseEstimator, TransformerMixin):
    """Advanced target transformations for optimal model performance"""
    
    def __init__(self, target_col='Cutoff_Rank'):
        self.target_col = target_col
        self.transformers = {}
        self.original_stats = {}
        self.transformation_stats = {}
        
    def fit(self, X, y=None):
        """Fit multiple transformation strategies"""
        
        if self.target_col not in X.columns:
            print(f"❌ Target column {self.target_col} not found")
            return self
        
        target = X[self.target_col].copy()
        
        # Store original statistics
        self.original_stats = {
            'mean': target.mean(),
            'std': target.std(),
            'skewness': target.skew(),
            'min': target.min(),
            'max': target.max()
        }
        
        print(f"   Original target statistics:")
        print(f"     Skewness: {self.original_stats['skewness']:.3f}")
        print(f"     Range: {self.original_stats['min']:,.0f} - {self.original_stats['max']:,.0f}")
        
        # 1. Log transformation (for right-skewed data)
        if target.min() > 0:
            log_target = np.log1p(target)  # log1p handles zeros better
            self.transformers['log1p'] = {
                'name': 'Log1p Transform',
                'function': np.log1p,
                'inverse': np.expm1,
                'skewness': log_target.skew()
            }
        
        # 2. Square root transformation
        if target.min() >= 0:
            sqrt_target = np.sqrt(target)
            self.transformers['sqrt'] = {
                'name': 'Square Root Transform',
                'function': np.sqrt,
                'inverse': np.square,
                'skewness': sqrt_target.skew()
            }
        
        # 3. Box-Cox transformation
        try:
            if target.min() > 0:
                # Find optimal lambda
                fitted_data, fitted_lambda = stats.boxcox(target)
                self.transformers['boxcox'] = {
                    'name': 'Box-Cox Transform',
                    'lambda': fitted_lambda,
                    'skewness': pd.Series(fitted_data).skew()
                }
        except:
            pass
        
        # 4. Yeo-Johnson transformation (handles negative values)
        try:
            pt = PowerTransformer(method='yeo-johnson', standardize=False)
            yj_target = pt.fit_transform(target.values.reshape(-1, 1)).ravel()
            self.transformers['yeo_johnson'] = {
                'name': 'Yeo-Johnson Transform',
                'transformer': pt,
                'skewness': pd.Series(yj_target).skew()
            }
        except:
            pass
        
        # 5. Quantile transformation (to normal)
        try:
            qt_normal = QuantileTransformer(output_distribution='normal', n_quantiles=1000)
            qt_normal_target = qt_normal.fit_transform(target.values.reshape(-1, 1)).ravel()
            self.transformers['quantile_normal'] = {
                'name': 'Quantile-to-Normal Transform',
                'transformer': qt_normal,
                'skewness': pd.Series(qt_normal_target).skew()
            }
        except:
            pass
        
        # 6. Quantile transformation (to uniform)
        try:
            qt_uniform = QuantileTransformer(output_distribution='uniform', n_quantiles=1000)
            qt_uniform_target = qt_uniform.fit_transform(target.values.reshape(-1, 1)).ravel()
            self.transformers['quantile_uniform'] = {
                'name': 'Quantile-to-Uniform Transform',
                'transformer': qt_uniform,
                'skewness': pd.Series(qt_uniform_target).skew()
            }
        except:
            pass
        
        # Evaluate transformations
        print(f"\n   TRANSFORMATION EVALUATION:")
        print(f"   {'Method':<25} {'Skewness':>10} {'Improvement'}")
        print("-" * 50)
        
        original_skew = abs(self.original_stats['skewness'])
        best_transform = None
        best_skew_improvement = 0
        
        for name, info in self.transformers.items():
            transform_skew = abs(info['skewness'])
            improvement = original_skew - transform_skew
            improvement_pct = (improvement / original_skew * 100) if original_skew > 0 else 0
            
            print(f"   {info['name']:<25} {info['skewness']:>10.3f} {improvement_pct:>8.1f}%")
            
            if improvement > best_skew_improvement:
                best_skew_improvement = improvement
                best_transform = name
        
        # Store best transformation
        self.best_transform = best_transform
        if best_transform:
            print(f"\n   ✅ Best transformation: {self.transformers[best_transform]['name']}")
            print(f"   ✅ Skewness improvement: {best_skew_improvement:.3f}")
        
        return self
    
    def transform(self, X):
        """Apply the best transformation"""
        X_transformed = X.copy()
        
        if self.target_col not in X_transformed.columns:
            return X_transformed
        
        target = X_transformed[self.target_col].copy()
        
        # Create raw target backup
        X_transformed[f'{self.target_col}_raw'] = target.copy()
        
        # Apply transformations
        for transform_name, transform_info in self.transformers.items():
            new_col_name = f'{self.target_col}_{transform_name}'
            
            try:
                if transform_name == 'log1p':
                    X_transformed[new_col_name] = np.log1p(target)
                elif transform_name == 'sqrt':
                    X_transformed[new_col_name] = np.sqrt(target)
                elif transform_name == 'boxcox':
                    lambda_val = transform_info['lambda']
                    if lambda_val == 0:
                        X_transformed[new_col_name] = np.log(target)
                    else:
                        X_transformed[new_col_name] = (target**lambda_val - 1) / lambda_val
                elif transform_name in ['yeo_johnson', 'quantile_normal', 'quantile_uniform']:
                    transformer = transform_info['transformer']
                    X_transformed[new_col_name] = transformer.transform(target.values.reshape(-1, 1)).ravel()
            except Exception as e:
                print(f"   ⚠️ Could not apply {transform_name}: {str(e)}")
        
        # Set the best transformation as primary target if available
        if self.best_transform and f'{self.target_col}_{self.best_transform}' in X_transformed.columns:
            X_transformed[f'{self.target_col}_transformed'] = X_transformed[f'{self.target_col}_{self.best_transform}']
            print(f"   ✅ Applied best transformation: {self.transformers[self.best_transform]['name']}")
        
        return X_transformed
    
    def get_transformation_summary(self):
        """Get summary of all transformations"""
        summary = []
        for name, info in self.transformers.items():
            summary.append({
                'transformation': info['name'],
                'method': name,
                'skewness': info['skewness'],
                'abs_skewness': abs(info['skewness']),
                'is_best': name == self.best_transform
            })
        return pd.DataFrame(summary)

# ============================================================================
# 3. APPLY TARGET TRANSFORMATIONS
# ============================================================================

print(f"\n🚀 STEP 2.3.3: APPLYING TARGET TRANSFORMATIONS")
print("="*60)

# Initialize and apply target transformer
target_transformer = AdvancedTargetTransformer(target_col='Cutoff_Rank')
df_transformed = target_transformer.fit_transform(df_imputed_final)

print(f"\nTarget transformations applied!")
print(f"Dataset shape: {df_imputed_final.shape} → {df_transformed.shape}")

# Show transformation summary
transform_summary = target_transformer.get_transformation_summary()
print(f"\nTRANSFORMATION SUMMARY:")
print(transform_summary.to_string(index=False))

# ============================================================================
# 4. ALTERNATIVE TARGET TASKS
# ============================================================================

print(f"\n🎯 STEP 2.3.4: ALTERNATIVE TARGET TASKS")
print("="*60)

class AlternativeTargetCreator(BaseEstimator, TransformerMixin):
    """Create alternative prediction tasks"""
    
    def __init__(self, target_col='Cutoff_Rank'):
        self.target_col = target_col
        self.percentile_thresholds = {}
        self.difficulty_thresholds = {}
        
    def fit(self, X, y=None):
        """Fit alternative target creators"""
        
        if self.target_col not in X.columns:
            return self
        
        target = X[self.target_col].copy()
        
        # Calculate percentile thresholds
        self.percentile_thresholds = {
            'very_competitive': target.quantile(0.1),   # Top 10%
            'competitive': target.quantile(0.25),       # Top 25%
            'moderate': target.quantile(0.5),           # Top 50%
            'accessible': target.quantile(0.75),        # Top 75%
        }
        
        # Difficulty thresholds based on domain knowledge
        self.difficulty_thresholds = {
            'elite': 10000,      # Very hard to get
            'good': 50000,       # Moderately hard
            'average': 100000,   # Average difficulty
            'basic': 200000      # Easy to get
        }
        
        print(f"   ✅ Percentile thresholds calculated:")
        for category, threshold in self.percentile_thresholds.items():
            print(f"     {category}: {threshold:,.0f}")
        
        return self
    
    def transform(self, X):
        """Create alternative target tasks"""
        X_alt = X.copy()
        
        if self.target_col not in X_alt.columns:
            return X_alt
        
        target = X_alt[self.target_col].copy()
        
        # 1. Binary classification tasks
        for category, threshold in self.percentile_thresholds.items():
            col_name = f'is_{category}'
            X_alt[col_name] = (target <= threshold).astype(int)
        
        # 2. Multi-class difficulty levels
        def classify_difficulty(rank):
            if rank <= self.difficulty_thresholds['elite']:
                return 'Elite'
            elif rank <= self.difficulty_thresholds['good']:
                return 'Good'
            elif rank <= self.difficulty_thresholds['average']:
                return 'Average'
            else:
                return 'Basic'
        
        X_alt['difficulty_category'] = target.apply(classify_difficulty)
        
        # 3. Percentile ranking (0-100 scale)
        X_alt['cutoff_percentile'] = target.rank(pct=True) * 100
        
        # 4. Log-scaled ranking for easier modeling
        X_alt['cutoff_rank_scaled'] = target / 1000  # Scale to thousands
        
        # 5. Relative ranking within groups
        if 'Branch' in X_alt.columns:
            X_alt['rank_within_branch'] = target.groupby(X_alt['Branch']).rank(pct=True)
        
        if 'Category' in X_alt.columns:
            X_alt['rank_within_category'] = target.groupby(X_alt['Category']).rank(pct=True)
        
        # 6. Success probability (inverse of rank, normalized)
        max_rank = target.max()
        X_alt['success_probability'] = (max_rank - target) / max_rank
        
        print(f"   ✅ Created alternative targets:")
        print(f"     Binary classification: 4 tasks")
        print(f"     Multi-class difficulty: 4 levels")
        print(f"     Percentile ranking: 0-100 scale")
        print(f"     Relative rankings: branch & category")
        print(f"     Success probability: 0-1 scale")
        
        return X_alt

# Apply alternative target creation
alt_target_creator = AlternativeTargetCreator(target_col='Cutoff_Rank')
df_final = alt_target_creator.fit_transform(df_transformed)

# ============================================================================
# 5. VALIDATION & SUMMARY
# ============================================================================

print(f"\n📊 STEP 2.3.5: VALIDATION & SUMMARY")
print("="*60)

print(f"TARGET ENGINEERING RESULTS:")
print(f"   Original dataset: {df_imputed_final.shape}")
print(f"   Final dataset: {df_final.shape}")
print(f"   New features created: {df_final.shape[1] - df_imputed_final.shape[1]}")

# Validate key transformations
print(f"\nKEY TRANSFORMATIONS VALIDATION:")

# Check original vs transformed target
if 'Cutoff_Rank_transformed' in df_final.columns:
    orig_skew = df_final['Cutoff_Rank'].skew()
    trans_skew = df_final['Cutoff_Rank_transformed'].skew()
    print(f"   Original skewness: {orig_skew:.3f}")
    print(f"   Transformed skewness: {trans_skew:.3f}")
    print(f"   Improvement: {abs(orig_skew) - abs(trans_skew):.3f}")

# Check alternative targets
alt_targets = ['is_very_competitive', 'is_competitive', 'difficulty_category', 'success_probability']
print(f"\nALTERNATIVE TARGET VALIDATION:")
for target in alt_targets:
    if target in df_final.columns:
        if df_final[target].dtype in ['int64', 'float64']:
            stats = df_final[target].describe()
            print(f"   {target}: range={stats['min']:.3f}-{stats['max']:.3f}, mean={stats['mean']:.3f}")
        else:
            counts = df_final[target].value_counts()
            print(f"   {target}: {len(counts)} categories, top={counts.index[0]}")

# ============================================================================
# 6. FINAL SUMMARY
# ============================================================================

print(f"\n✅ STEP 2.3: TARGET TRANSFORMATIONS COMPLETE!")
print("="*80)

print(f"ACCOMPLISHED:")
print(f"   ✅ Comprehensive target distribution analysis")
print(f"   ✅ Multiple transformation strategies evaluated")
print(f"   ✅ Best transformation automatically selected")
print(f"   ✅ Alternative prediction tasks created")
print(f"   ✅ Binary classification targets")
print(f"   ✅ Multi-class difficulty levels")
print(f"   ✅ Relative ranking features")
print(f"   ✅ Success probability modeling")

print(f"\nFINAL DATASET FOR MODELING:")
print(f"   📊 Shape: {df_final.shape}")
print(f"   🎯 Primary target: Cutoff_Rank (original)")
print(f"   🔄 Transformed target: Cutoff_Rank_transformed (best)")
print(f"   📈 Alternative targets: {len([c for c in df_final.columns if c.startswith('is_') or 'difficulty' in c])}")
print(f"   ⚙️ Ready for Step 2.4: Advanced Lag Features")

print(f"\n🚀 READY FOR STEP 2.4: LAG & HISTORICAL AGGREGATION FEATURES!")

# Store final dataset
df_target_engineered = df_final.copy()


STAGE 2 - STEP 2.3: TARGET TRANSFORMATIONS & ALTERNATIVE TASKS
Starting with imputed data: 270,062 records, 29 features

🔍 STEP 2.3.1: TARGET VARIABLE COMPREHENSIVE ANALYSIS
TARGET VARIABLE ANALYSIS (Cutoff_Rank):
   Count: 270,062
   Mean: 82,766
   Median: 72,236
   Std Dev: 56,070
   Range: 90 - 274,884
   IQR: 78,708
   Skewness: 0.836
   Kurtosis: 0.298

DISTRIBUTION CHARACTERISTICS:
   Skewness: Moderate (Right skewed)
   Kurtosis: Platykurtic (flatter)
   Outliers: 4,242 (1.57%)
   Normal range: -79,557 - 235,274

🔧 STEP 2.3.2: TARGET TRANSFORMATION STRATEGIES

🚀 STEP 2.3.3: APPLYING TARGET TRANSFORMATIONS
   Original target statistics:
     Skewness: 0.836
     Range: 90 - 274,884

   TRANSFORMATION EVALUATION:
   Method                      Skewness Improvement
--------------------------------------------------
   Log1p Transform               -1.237    -48.0%
   Square Root Transform          0.044     94.7%
   Box-Cox Transform             -0.073     91.2%
   Yeo-Johnson Tra

In [6]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
import time

PRIMARY_GROUPS = ['College_Code', 'Branch', 'Category']
SECONDARY_GROUPS = ['College_Code', 'Branch']
TERTIARY_GROUPS = ['Branch']
TIME_COLUMN = 'Year'
TARGET_COLUMN = 'Cutoff_Rank'
THRESHOLD_MIN_GROUPS = 10  # Lower threshold for lag feature generation

class AdvancedLagFeatureTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, group_cols_list, time_col, target_col, lag_periods=[1,2,3], rolling_windows=[2,3,5], min_periods=1):
        self.group_cols_list = group_cols_list
        self.time_col = time_col
        self.target_col = target_col
        self.lag_periods = lag_periods
        self.rolling_windows = rolling_windows
        self.min_periods = min_periods
        self.feature_stats = {}
        self.generated_features = []

    def fit(self, X, y=None):
        print("-" * 60)
        print("Analyzing temporal patterns for lag features...")
        for i, group_cols in enumerate(self.group_cols_list):
            group_name = f"L{i+1}"
            if not all(col in X.columns for col in group_cols):
                print(f"⚠️ Skipping {group_name}: missing columns")
                continue
            group_coverage = X.groupby(group_cols)[self.time_col].nunique()
            sufficient_history = (group_coverage >= max(self.lag_periods)+1).sum()
            print(f"{group_name} ({'+'.join(group_cols)}): {sufficient_history:,} groups with sufficient history")
            self.feature_stats[group_name] = {
                'group_cols': group_cols,
                'sufficient_history': sufficient_history,
                'total_groups': len(group_coverage)
            }
        print("✅ Temporal analysis complete for", len(self.feature_stats), "grouping levels")
        return self

    def transform(self, X):
        print("Generating lag features for each grouping (progress shown for each level)")
        X_lagged = X.copy()
        total_start = time.time()
        for group_name, group_info in self.feature_stats.items():
            group_cols = group_info['group_cols']
            if group_info['sufficient_history'] < THRESHOLD_MIN_GROUPS:
                print(f"\n⚠️ Skipping {group_name}: only {group_info['sufficient_history']}/{THRESHOLD_MIN_GROUPS} groups with sufficient history.")
                continue
            print(f"\nProcessing {group_name} ({'+'.join(group_cols)})...")
            grouped = X_lagged.groupby(group_cols)[self.target_col]
            t0 = time.time()
            for lag in self.lag_periods:
                col = f"{self.target_col}_lag{lag}_{group_name}"
                X_lagged[col] = grouped.shift(lag)
                self.generated_features.append(col)
                print(f"  - {col} ✅")
            for w in self.rolling_windows:
                if w <= max(self.lag_periods):
                    mean_col = f"{self.target_col}_rollmean{w}_{group_name}"
                    std_col  = f"{self.target_col}_rollstd{w}_{group_name}"
                    shifted = grouped.shift(1)
                    X_lagged[mean_col] = shifted.rolling(w, min_periods=self.min_periods).mean()
                    X_lagged[std_col] = shifted.rolling(w, min_periods=self.min_periods).std()
                    self.generated_features += [mean_col, std_col]
                    print(f"  - {mean_col}, {std_col} ✅")
            print(f" ---> {group_name} done. Time: {int(time.time() - t0)}s")
        print("\nAll feature lags/rollings done. Total elapsed:", int(time.time() - total_start), "s")
        return X_lagged

    def get_feature_summary(self):
        return {
            'total_features': len(self.generated_features),
            'feature_names': self.generated_features[:20],
            'grouping_levels': list(self.feature_stats.keys())
        }

# --- Usage example ---

grouping_strategies = [PRIMARY_GROUPS, SECONDARY_GROUPS, TERTIARY_GROUPS]
lag_transformer = AdvancedLagFeatureTransformer(
    grouping_strategies,
    time_col=TIME_COLUMN,
    target_col=TARGET_COLUMN,
    lag_periods=[1, 2, 3],
    rolling_windows=[2, 3, 5]
)

print("\nRunning lag feature engineering (watch progress and ETA)...")
tfit = time.time()
lag_transformer.fit(df_target_engineered)
print(f"Fit completed in {int(time.time() - tfit)} seconds\n")

tstart = time.time()
df_with_lags = lag_transformer.transform(df_target_engineered)
tend = time.time()
print(f"\nLag feature generation DONE. Total elapsed: {int((tend - tstart) // 60)}m{int((tend - tstart) % 60)}s.")

feature_summary = lag_transformer.get_feature_summary()
print(f"Total lag features created: {feature_summary['total_features']}")
print(f"Grouping levels used: {feature_summary['grouping_levels']}")

print(f"\n--- Step 2.4 complete, with progress and ETA info. ---")



Running lag feature engineering (watch progress and ETA)...
------------------------------------------------------------
Analyzing temporal patterns for lag features...
L1 (College_Code+Branch+Category): 4,666 groups with sufficient history
L2 (College_Code+Branch): 916 groups with sufficient history
L3 (Branch): 18 groups with sufficient history
✅ Temporal analysis complete for 3 grouping levels
Fit completed in 0 seconds

Generating lag features for each grouping (progress shown for each level)

Processing L1 (College_Code+Branch+Category)...
  - Cutoff_Rank_lag1_L1 ✅
  - Cutoff_Rank_lag2_L1 ✅
  - Cutoff_Rank_lag3_L1 ✅
  - Cutoff_Rank_rollmean2_L1, Cutoff_Rank_rollstd2_L1 ✅
  - Cutoff_Rank_rollmean3_L1, Cutoff_Rank_rollstd3_L1 ✅
 ---> L1 done. Time: 0s

Processing L2 (College_Code+Branch)...
  - Cutoff_Rank_lag1_L2 ✅
  - Cutoff_Rank_lag2_L2 ✅
  - Cutoff_Rank_lag3_L2 ✅
  - Cutoff_Rank_rollmean2_L2, Cutoff_Rank_rollstd2_L2 ✅
  - Cutoff_Rank_rollmean3_L2, Cutoff_Rank_rollstd3_L2 ✅
 ---

In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STAGE 2 - STEP 2.5: AGGREGATE & CROSS-GROUP FEATURES")
print("=" * 80)

# Use output of Step 2.4 lag feature engineering as input
df = df_with_lags.copy()

# ==============================================================================
# 1. COLLEGE-WIDE AGGREGATES (excluding current year)
# ==============================================================================

print(f"\n🔧 STEP 2.5.1: COLLEGE-WIDE AGGREGATES")
print("=" * 60)

df = df.sort_values(['College_Code', 'Year'])

df['college_avg_cutoff_branch_last_year'] = (
    df.groupby(['College_Code', 'Branch'])['Cutoff_Rank']
    .shift(1)
    .transform(lambda x: x.rolling(window=1, min_periods=1).mean())
)

df['college_cutoff_std_allbranches_past'] = (
    df.groupby('College_Code')['Cutoff_Rank']
    .shift(1)
    .transform(lambda x: x.expanding(min_periods=1).std())
)

print("Created features: college_avg_cutoff_branch_last_year, college_cutoff_std_allbranches_past")

# ==============================================================================
# 2. BRANCH-WIDE AGGREGATES ACROSS COLLEGES
# ==============================================================================

print(f"\n🔧 STEP 2.5.2: BRANCH-WIDE AGGREGATES")
print("=" * 60)

df = df.sort_values(['Branch', 'Year'])

df['branch_global_avg_cutoff_last_year'] = (
    df.groupby('Branch')['Cutoff_Rank']
    .shift(1)
    .transform(lambda x: x.expanding(min_periods=1).mean())
)

df['branch_volatility'] = (
    df.groupby('Branch')['Cutoff_Rank']
    .shift(1)
    .transform(lambda x: x.rolling(window=3, min_periods=1).std())
)

print("Created features: branch_global_avg_cutoff_last_year, branch_volatility")

# ==============================================================================
# 3. CATEGORY SHARE PROXIES
# ==============================================================================

print(f"\n🔧 STEP 2.5.3: CATEGORY SHARE PROXIES")
print("=" * 60)

df = df.sort_values(['College_Code', 'Branch', 'Year'])

category_mean = (
    df.groupby(['College_Code', 'Branch', 'Category'])['Cutoff_Rank']
    .transform('mean')
)

all_cat_mean = (
    df.groupby(['College_Code', 'Branch'])['Cutoff_Rank']
    .transform('mean')
)

df['category_ratio'] = category_mean / all_cat_mean
df['category_ratio'] = df['category_ratio'].replace([np.inf, -np.inf], np.nan).fillna(1.0)

print("Created feature: category_ratio")

# ==============================================================================
# 4. SUMMARY AND READY FOR NEXT STEP
# ==============================================================================

print(f"\n✅ STEP 2.5: AGGREGATE & CROSS-GROUP FEATURES COMPLETE!")
print(f"   Dataset shape: {df_with_lags.shape} → {df.shape}")
print(f"   New features added:")

new_features = [
    'college_avg_cutoff_branch_last_year', 
    'college_cutoff_std_allbranches_past',
    'branch_global_avg_cutoff_last_year',
    'branch_volatility',
    'category_ratio'
]

for nf in new_features:
    if nf in df.columns:
        print(f"    - {nf}")

# Store for next step
df_agg_cross = df.copy()

print(f"\n🚀 READY FOR STEP 2.6: HIGH-CARDINALITY ENCODING!")


STAGE 2 - STEP 2.5: AGGREGATE & CROSS-GROUP FEATURES

🔧 STEP 2.5.1: COLLEGE-WIDE AGGREGATES
Created features: college_avg_cutoff_branch_last_year, college_cutoff_std_allbranches_past

🔧 STEP 2.5.2: BRANCH-WIDE AGGREGATES
Created features: branch_global_avg_cutoff_last_year, branch_volatility

🔧 STEP 2.5.3: CATEGORY SHARE PROXIES
Created feature: category_ratio

✅ STEP 2.5: AGGREGATE & CROSS-GROUP FEATURES COMPLETE!
   Dataset shape: (270062, 68) → (270062, 73)
   New features added:
    - college_avg_cutoff_branch_last_year
    - college_cutoff_std_allbranches_past
    - branch_global_avg_cutoff_last_year
    - branch_volatility
    - category_ratio

🚀 READY FOR STEP 2.6: HIGH-CARDINALITY ENCODING!


In [8]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STAGE 2 - STEP 2.6: HIGH-CARDINALITY ENCODING")
print("=" * 80)

# Use output from Step 2.5
df = df_agg_cross.copy()
print(f"Starting with: {df.shape[0]:,} records, {df.shape[1]} features")

# ============================================================================
# 1. ANALYZE HIGH-CARDINALITY FEATURES
# ============================================================================

print(f"\n🔍 STEP 2.6.1: HIGH-CARDINALITY ANALYSIS")
print("=" * 60)

def analyze_cardinality(df, threshold=10):
    """Analyze cardinality of categorical features"""
    categorical_cols = df.select_dtypes(include=['object']).columns
    cardinality_info = {}
    
    for col in categorical_cols:
        unique_count = df[col].nunique()
        total_records = len(df)
        cardinality_ratio = unique_count / total_records
        
        cardinality_info[col] = {
            'unique_count': unique_count,
            'cardinality_ratio': cardinality_ratio,
            'encoding_strategy': 'high_card' if unique_count > threshold else 'standard'
        }
    
    return cardinality_info

cardinality_analysis = analyze_cardinality(df, threshold=20)

print("CARDINALITY ANALYSIS:")
for col, info in cardinality_analysis.items():
    print(f"   {col}: {info['unique_count']} unique ({info['cardinality_ratio']:.3%}) - {info['encoding_strategy']}")

# ============================================================================
# 2. TARGET ENCODING WITH SMOOTHING
# ============================================================================

print(f"\n🔧 STEP 2.6.2: TARGET ENCODING WITH SMOOTHING")
print("=" * 60)

class SmoothTargetEncoder(BaseEstimator, TransformerMixin):
    """Target encoder with smoothing to prevent overfitting"""
    
    def __init__(self, categorical_cols, target_col, smoothing=100, noise_level=0.01):
        self.categorical_cols = categorical_cols
        self.target_col = target_col
        self.smoothing = smoothing
        self.noise_level = noise_level
        self.encodings = {}
        self.global_mean = 0
        
    def fit(self, X, y=None):
        """Fit encoder with smoothed target encoding"""
        if self.target_col not in X.columns:
            print(f"   ⚠️ Target column {self.target_col} not found")
            return self
            
        self.global_mean = X[self.target_col].mean()
        
        for col in self.categorical_cols:
            if col in X.columns:
                # Calculate smoothed target encoding
                agg = X.groupby(col)[self.target_col].agg(['mean', 'count']).reset_index()
                
                # Smoothing formula: (count * mean + smoothing * global_mean) / (count + smoothing)
                agg['smoothed_mean'] = (
                    (agg['count'] * agg['mean'] + self.smoothing * self.global_mean) / 
                    (agg['count'] + self.smoothing)
                )
                
                # Add small random noise to prevent overfitting
                noise = np.random.normal(0, self.noise_level * self.global_mean, len(agg))
                agg['smoothed_mean'] += noise
                
                self.encodings[col] = dict(zip(agg[col], agg['smoothed_mean']))
                print(f"   ✅ {col}: {len(agg)} categories encoded")
        
        return self
    
    def transform(self, X):
        """Apply target encoding"""
        X_encoded = X.copy()
        
        for col in self.categorical_cols:
            if col in X_encoded.columns and col in self.encodings:
                encoded_col = f"{col}_target_enc"
                X_encoded[encoded_col] = X_encoded[col].map(self.encodings[col]).fillna(self.global_mean)
                
        return X_encoded

# Apply target encoding to high-cardinality features
high_card_cols = [col for col, info in cardinality_analysis.items() 
                  if info['encoding_strategy'] == 'high_card']

if high_card_cols:
    target_encoder = SmoothTargetEncoder(
        categorical_cols=high_card_cols,
        target_col='Cutoff_Rank',
        smoothing=50,  # Lower smoothing for more responsiveness
        noise_level=0.005
    )
    
    df = target_encoder.fit_transform(df)
    print(f"Target encoding applied to: {high_card_cols}")

# ============================================================================
# 3. FREQUENCY ENCODING
# ============================================================================

print(f"\n🔧 STEP 2.6.3: FREQUENCY ENCODING")
print("=" * 60)

def add_frequency_encoding(df, categorical_cols):
    """Add frequency encoding for categorical variables"""
    for col in categorical_cols:
        if col in df.columns:
            freq_col = f"{col}_freq"
            frequency_map = df[col].value_counts().to_dict()
            df[freq_col] = df[col].map(frequency_map)
            print(f"   ✅ {col}: frequency encoding added")
    return df

# Apply frequency encoding
categorical_for_freq = ['College_Code', 'Branch', 'Category', 'Exam_Type']
available_cat_cols = [col for col in categorical_for_freq if col in df.columns]

df = add_frequency_encoding(df, available_cat_cols)

# ============================================================================
# 4. ORDINAL ENCODING FOR STRUCTURED CATEGORIES
# ============================================================================

print(f"\n🔧 STEP 2.6.4: ORDINAL ENCODING")
print("=" * 60)

# Ordinal mapping for categories with natural order
ordinal_mappings = {
    'College_Tier': {'Elite': 1, 'Very_Good': 2, 'Good': 3, 'Average': 4, 'Basic': 5},
    'Temporal_Regime': {'STABLE': 1, 'TRANSITION': 2, 'CRISIS': 3},
    'Market_Stress_Level': {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3}
}

for col, mapping in ordinal_mappings.items():
    if col in df.columns:
        ordinal_col = f"{col}_ordinal"
        df[ordinal_col] = df[col].map(mapping).fillna(0)
        print(f"   ✅ {col}: ordinal encoding applied")

# ============================================================================
# 5. BINARY ENCODING FOR MEDIUM CARDINALITY
# ============================================================================

print(f"\n🔧 STEP 2.6.5: BINARY ENCODING")
print("=" * 60)

def create_binary_features(df, col, top_n=10):
    """Create binary features for top N categories"""
    if col not in df.columns:
        return df
        
    top_categories = df[col].value_counts().head(top_n).index
    
    for category in top_categories:
        binary_col = f"{col}_is_{category}".replace(' ', '_').replace('&', 'and')
        df[binary_col] = (df[col] == category).astype(int)
    
    print(f"   ✅ {col}: {len(top_categories)} binary features created")
    return df

# Apply binary encoding to medium cardinality features
medium_card_cols = [col for col, info in cardinality_analysis.items() 
                   if 5 < info['unique_count'] <= 20]

for col in medium_card_cols:
    df = create_binary_features(df, col, top_n=min(10, cardinality_analysis[col]['unique_count']))

# ============================================================================
# 6. VALIDATION & SUMMARY
# ============================================================================

print(f"\n📊 STEP 2.6.6: ENCODING VALIDATION & SUMMARY")
print("=" * 60)

# Count new features by type
encoding_features = {
    'target_encoded': [col for col in df.columns if '_target_enc' in col],
    'frequency_encoded': [col for col in df.columns if '_freq' in col],
    'ordinal_encoded': [col for col in df.columns if '_ordinal' in col],
    'binary_encoded': [col for col in df.columns if '_is_' in col]
}

print("ENCODING SUMMARY:")
for enc_type, features in encoding_features.items():
    print(f"   {enc_type}: {len(features)} features")

# Check for any remaining high-cardinality categorical columns
remaining_categorical = df.select_dtypes(include=['object']).columns
high_card_remaining = [col for col in remaining_categorical 
                      if df[col].nunique() > 20]

if high_card_remaining:
    print(f"\n⚠️ Remaining high-cardinality columns: {high_card_remaining}")
else:
    print(f"\n✅ All high-cardinality columns properly encoded")

# ============================================================================
# 7. FINAL SUMMARY
# ============================================================================

print(f"\n✅ STEP 2.6: HIGH-CARDINALITY ENCODING COMPLETE!")
print("=" * 80)

print(f"ACCOMPLISHED:")
print(f"   ✅ Cardinality analysis performed")
print(f"   ✅ Smoothed target encoding with noise")
print(f"   ✅ Frequency encoding applied")
print(f"   ✅ Ordinal encoding for structured categories")
print(f"   ✅ Binary encoding for top categories")

print(f"\nFINAL DATASET STATISTICS:")
print(f"   📊 Shape: {df_agg_cross.shape} → {df.shape}")
print(f"   🔢 New encoding features: {df.shape[1] - df_agg_cross.shape[1]}")
print(f"   📈 Total features: {df.shape[1]}")

# Store for next step
df_encoded = df.copy()

print(f"\n🚀 READY FOR STEP 2.7: CATEGORICAL & INTERACTION FEATURES!")


STAGE 2 - STEP 2.6: HIGH-CARDINALITY ENCODING
Starting with: 270,062 records, 73 features

🔍 STEP 2.6.1: HIGH-CARDINALITY ANALYSIS
CARDINALITY ANALYSIS:
   College_Code: 261 unique (0.097%) - high_card
   College_Name: 652 unique (0.241%) - high_card
   Category: 8 unique (0.003%) - standard
   Branch: 33 unique (0.012%) - high_card
   Exam_Type: 2 unique (0.001%) - standard
   College_Tier: 5 unique (0.002%) - standard
   Category_Simplified: 5 unique (0.002%) - standard
   Volatility_Category: 3 unique (0.001%) - standard
   Program_Maturity: 3 unique (0.001%) - standard
   difficulty_category: 4 unique (0.001%) - standard

🔧 STEP 2.6.2: TARGET ENCODING WITH SMOOTHING
   ✅ College_Code: 261 categories encoded
   ✅ College_Name: 652 categories encoded
   ✅ Branch: 33 categories encoded
Target encoding applied to: ['College_Code', 'College_Name', 'Branch']

🔧 STEP 2.6.3: FREQUENCY ENCODING
   ✅ College_Code: frequency encoding added
   ✅ Branch: frequency encoding added
   ✅ Category: 

In [9]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STAGE 2 - STEP 2.7: CATEGORICAL & INTERACTION FEATURES")
print("=" * 80)

df = df_encoded.copy()
print(f"Starting with: {df.shape[0]:,} records, {df.shape[1]} features")

# ------------------------------------------------------------------------------
# 1. CROSS & INTERACTION FEATURES
# ------------------------------------------------------------------------------

print("\n🔧 Creating Pairwise and Triple Feature Interactions:")

# Pairwise
df['College_Branch'] = df['College_Code'].astype(str) + '__' + df['Branch'].astype(str)
df['Branch_Category'] = df['Branch'].astype(str) + '__' + df['Category'].astype(str)
df['College_Category'] = df['College_Code'].astype(str) + '__' + df['Category'].astype(str)
df['Branch_Tier'] = df['Branch'].astype(str) + '__' + df['College_Tier'].astype(str)

# Triple
df['College_Branch_Category'] = (df['College_Code'].astype(str) + '__' +
                                 df['Branch'].astype(str) + '__' +
                                 df['Category'].astype(str))

print("  - College_Branch, Branch_Category, College_Category, Branch_Tier")
print("  - College_Branch_Category (triple interaction)")

# Calculate the frequency of each triple combination (to detect rare/new patterns)
df['CBC_freq'] = df['College_Branch_Category'].map(df['College_Branch_Category'].value_counts())
print("  - CBC_freq (frequency of triple interaction)")

# ------------------------------------------------------------------------------
# 2. GROUP/INTERACTION TARGET ENCODINGS
# ------------------------------------------------------------------------------

print("\n🔧 Target Encoding on Key Interactions (Leakage-Free):")
for col in ['College_Branch', 'Branch_Category', 'College_Category', 'College_Branch_Category']:
    target_col = f'{col}_target_enc'
    avg = df.groupby(col)['Cutoff_Rank'].transform(lambda x: x.shift(1).expanding().mean())
    # Use overall mean for very first observation in every group
    avg.fillna(df['Cutoff_Rank'].mean(), inplace=True)
    df[target_col] = avg
    print(f"  - {target_col} created")

# ------------------------------------------------------------------------------
# 3. ENCODING VALIDATION & SUMMARY
# ------------------------------------------------------------------------------

print(f"\n📊 STEP 2.7: INTERACTION FEATURE SUMMARY")
print("=" * 60)

interaction_cols = [
    'College_Branch', 'Branch_Category', 'College_Category',
    'Branch_Tier', 'College_Branch_Category', 'CBC_freq'
] + [f'{col}_target_enc' for col in ['College_Branch', 'Branch_Category', 'College_Category', 'College_Branch_Category']]

print(f"  Total interaction/cross features: {len(interaction_cols)}")
for c in interaction_cols:
    if c in df.columns:
        print(f"    - {c}")

print(f"  Sample frequencies next 3 combos:\n{df['College_Branch_Category'].value_counts().head(3)}")

print(f"\n✅ STEP 2.7: CATEGORICAL & INTERACTION FEATURES COMPLETE!")
print("=" * 80)
print(f"ACCOMPLISHED:")
print(f"   ✅ Pairwise and triple group cross features")
print(f"   ✅ Interaction target encodings without leakage")
print(f"   ✅ Frequecy features exposing rare/new combos")

print(f"\nFINAL DATASET STATISTICS:")
print(f"   📊 Shape: {df_encoded.shape} → {df.shape}")
print(f"   📈 Total features: {df.shape[1]}")

# Store output for next module
df_interactions = df.copy()
print(f"\n🚀 READY FOR STEP 2.8: TEMPORAL & METADATA FEATURES!")


STAGE 2 - STEP 2.7: CATEGORICAL & INTERACTION FEATURES
Starting with: 270,062 records, 89 features

🔧 Creating Pairwise and Triple Feature Interactions:
  - College_Branch, Branch_Category, College_Category, Branch_Tier
  - College_Branch_Category (triple interaction)
  - CBC_freq (frequency of triple interaction)

🔧 Target Encoding on Key Interactions (Leakage-Free):
  - College_Branch_target_enc created
  - Branch_Category_target_enc created
  - College_Category_target_enc created
  - College_Branch_Category_target_enc created

📊 STEP 2.7: INTERACTION FEATURE SUMMARY
  Total interaction/cross features: 10
    - College_Branch
    - Branch_Category
    - College_Category
    - Branch_Tier
    - College_Branch_Category
    - CBC_freq
    - College_Branch_target_enc
    - Branch_Category_target_enc
    - College_Category_target_enc
    - College_Branch_Category_target_enc
  Sample frequencies next 3 combos:
College_Branch_Category
E237__Electronics__General    860
E232__Electronics__Gen

In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("STAGE 2 - STEP 2.8: TEMPORAL & METADATA FEATURES")
print("=" * 80)

df = df_interactions.copy()
print(f"Starting with: {df.shape[0]:,} records, {df.shape[1]} features")

# -------------------------------------------------------------------
# 1. TEMPORAL CONTEXT FEATURES
# -------------------------------------------------------------------

print("\n🔧 Adding Temporal Context Features:")

REFERENCE_YEAR = 2020
LATEST_YEAR = df['Year'].max()

# Years since reference (e.g., a key policy year)
df['years_since_reference'] = df['Year'] - REFERENCE_YEAR

# Is row from latest year?
df['is_latest_year'] = (df['Year'] == LATEST_YEAR).astype(int)

# Is "crisis year" (change as per domain/EDA)
df['is_crisis_year'] = (df['Year'] == 2024).astype(int)

# Pre- and post-crisis indicators
df['pre_crisis'] = (df['Year'] < 2024).astype(int)
df['post_crisis'] = (df['Year'] > 2024).astype(int)

# Cohort/era as a categorical feature (adjust bins/labels if needed)
df['year_cohort'] = pd.cut(df['Year'], bins=[2019,2022,2024,2050], labels=['Stable','Transition','Crisis'])

print("  - years_since_reference, is_latest_year, is_crisis_year, pre_crisis, post_crisis, year_cohort")

# -------------------------------------------------------------------
# 2. METADATA & DATA QUALITY SIGNALS
# -------------------------------------------------------------------

print("\n🔧 Adding Metadata & Data Quality Features:")

# New program indicator from history, if available
if 'years_history_L1' in df.columns:
    df['is_new_program'] = (df['years_history_L1'] == 0).astype(int)
    print("  - is_new_program")

# Low-history program indicator
if 'years_history_L1' in df.columns:
    df['is_low_history_program'] = (df['years_history_L1'] <= 2).astype(int)
    print("  - is_low_history_program")

# Recency: proportion of years between oldest and latest
df['relative_recency'] = (df['Year'] - df['Year'].min()) / (LATEST_YEAR - df['Year'].min())

# Unique ID for row-level tracking (optional, drop for modeling)
df['unique_row_id'] = (
    df['College_Code'].astype(str) + "__" +
    df['Branch'].astype(str) + "__" +
    df['Category'].astype(str) + "__" +
    df['Year'].astype(str)
)

print("  - relative_recency, unique_row_id")

# -------------------------------------------------------------------
# 3. FINALIZE
# -------------------------------------------------------------------

print(f"\n✅ STEP 2.8 COMPLETE! Dataset: {df.shape}")
added = [
    'years_since_reference','is_latest_year','is_crisis_year','pre_crisis','post_crisis',
    'year_cohort','is_new_program','is_low_history_program','relative_recency','unique_row_id'
]
print("   New features added:")
for c in added:
    if c in df.columns:
        print(f"    - {c}")

df_temporal_meta = df.copy()
print("\n🚀 READY FOR STEP 2.9: HANDLE LOW-HISTORY COLLEGES!")


STAGE 2 - STEP 2.8: TEMPORAL & METADATA FEATURES
Starting with: 270,062 records, 99 features

🔧 Adding Temporal Context Features:
  - years_since_reference, is_latest_year, is_crisis_year, pre_crisis, post_crisis, year_cohort

🔧 Adding Metadata & Data Quality Features:
  - relative_recency, unique_row_id

✅ STEP 2.8 COMPLETE! Dataset: (270062, 107)
   New features added:
    - years_since_reference
    - is_latest_year
    - is_crisis_year
    - pre_crisis
    - post_crisis
    - year_cohort
    - relative_recency
    - unique_row_id

🚀 READY FOR STEP 2.9: HANDLE LOW-HISTORY COLLEGES!
